# Palmer Amaranth Weed Detection — Prithvi EO 2.0 + Weed-Intensity Map

Local Jupyter. Data (`training/` and `validation/`) is already extracted next to this notebook.
Run top to bottom. Cell 1 fixes packages, then RESTART THE KERNEL and skip Cell 1.

## Cell 1 — Fix packages (run ONCE, then restart kernel)

TerraTorch 0.99.8 needs segmentation-models-pytorch 0.3.4 and its two deps.
After this finishes, **Kernel -> Restart**, then SKIP this cell and start at Cell 2.

In [ ]:
%pip install "segmentation-models-pytorch==0.3.4" --force-reinstall --no-deps
%pip install pretrainedmodels efficientnet-pytorch

## Cell 2 — Imports

In [1]:
import numpy as np
import torch
import albumentations
import albumentations.pytorch
import lightning.pytorch as pl
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm

import terratorch
from terratorch.datamodules import GenericNonGeoSegmentationDataModule
from terratorch.tasks import SemanticSegmentationTask

import segmentation_models_pytorch as smp
print("smp version:", smp.__version__, "(must be 0.3.4)")
print("CUDA available:", torch.cuda.is_available())

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations


smp version: 0.3.4 (must be 0.3.4)
CUDA available: False


## Cell 3 — Data module

Data is already extracted at `training/` and `validation/` next to this notebook.
`num_workers=0` for Windows stability. If this prints `Train: 0 Val: 0`,
the grep patterns don't match the filenames — check one filename in `training/`.

In [2]:
datamodule = GenericNonGeoSegmentationDataModule(
    batch_size=8,
    num_workers=0,
    num_classes=2,

    train_data_root='training/',
    train_label_data_root='training/',
    val_data_root='validation/',
    val_label_data_root='validation/',
    test_data_root='validation/',
    test_label_data_root='validation/',

    img_grep='*_merged.tif',
    label_grep='*.mask.tif',

    train_transform=[
        albumentations.D4(),
        albumentations.pytorch.transforms.ToTensorV2(),
    ],
    val_transform=None,
    test_transform=None,

    means=[0.0333497067415863, 0.0570118552053618, 0.0588974813200132,
           0.2323245113436119, 0.1972854853760658, 0.1194491422518656],
    stds=[0.0226913556882377, 0.0268075602230702, 0.0400410984436278,
          0.0779173242367269, 0.0870873883814014, 0.0724197947743781],
)

datamodule.setup("fit")
print("Train:", len(datamodule.train_dataset), "Val:", len(datamodule.val_dataset))

Train: 798 Val: 200


## Cell 4 — Model + trainer (frozen Prithvi backbone + UNet decoder, 5 epochs)

In [3]:
pl.seed_everything(0)

checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath="output/weed/checkpoints/",
    mode="max",
    monitor="val/Multiclass_Jaccard_Index",
    filename="best-{epoch:02d}",
)

trainer = pl.Trainer(
    accelerator="auto",
    strategy="auto",
    devices=1,
    precision="32-true",
    num_nodes=1,
    logger=True,
    max_epochs=5,
    log_every_n_steps=1,
    enable_checkpointing=True,
    callbacks=[checkpoint_callback, pl.callbacks.RichProgressBar()],
    default_root_dir="output/weed",
)

model = SemanticSegmentationTask(
    model_factory="EncoderDecoderFactory",
    model_args={
        "backbone": "prithvi_eo_v2_300_tl",
        "backbone_pretrained": True,
        "backbone_num_frames": 1,
        "backbone_bands": ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"],
        "backbone_coords_encoding": [],
        "necks": [
            {"name": "SelectIndices", "indices": [5, 11, 17, 23]},
            {"name": "ReshapeTokensToImage"},
            {"name": "LearnedInterpolateToPyramidal"},
        ],
        "decoder": "UNetDecoder",
        "decoder_channels": [512, 256, 128, 64],
        "head_dropout": 0.1,
        "num_classes": 2,
    },
    loss="dice",
    optimizer="AdamW",
    lr=1e-4,
    ignore_index=-1,
    freeze_backbone=True,
    freeze_decoder=False,
    plot_on_val=True,
    class_names=["no weed", "weed"],
)

INFO: Seed set to 0
INFO:lightning.fabric.utilities.seed:Seed set to 0
INFO: GPU available: False, used: False
INFO:lightning.pytorch.utilities.rank_zero:GPU available: False, used: False
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: HPU available: False, using: 0 HPUs
INFO:lightning.pytorch.utilities.rank_zero:HPU available: False, using: 0 HPUs
INFO:httpx:HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/63adbd39c271da4c42f447e69b1a7c91a338cdc9/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/ibm-nasa-geospatial/Prithvi-EO-2.0-300M-TL/resolve/main/Prithvi_EO_V2_300M_TL.pt "HTTP/1.1 302 Found"
INFO:root:Loaded weights for HLSBands.BLUE in position 0 o

## Cell 5 — Train

In [ ]:
trainer.fit(model, datamodule=datamodule)

┏━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name          ┃ Type             ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ model         │ PixelWiseModel   │  324 M │ train │
│ 1 │ criterion     │ DiceLoss         │      0 │ train │
│ 2 │ train_metrics │ MetricCollection │      0 │ train │
│ 3 │ val_metrics   │ MetricCollection │      0 │ train │
│ 4 │ test_metrics  │ ModuleList       │      0 │ train │
└───┴───────────────┴──────────────────┴────────┴───────┘

Trainable params: 20.3 M                                                                                           
Non-trainable params: 303 M                                                                                        
Total params: 324 M                                                                                                
Total estimated model params size (MB): 1.3 K                                                                      
Modules in train mode: 641                                                                                         
Modules in eval mode: 0

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:4
25: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the
`num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

C:\Users\aishv\anaconda3\envs\terratorch\Lib\site-packages\rich\live.py:231: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

## Cell 6 — Test

Expected: micro IoU ~86%, weed IoU ~0. Near-zero weed IoU is EXPECTED with fake labels.
The point is the pipeline runs end to end.

In [ ]:
best_ckpt_path = checkpoint_callback.best_model_path
print(best_ckpt_path)

datamodule.setup("test")
trainer.test(model, datamodule=datamodule, ckpt_path=best_ckpt_path)

## Cell 7 — Weed-Intensity Map (probability -> grid cells -> severity bins)

Per pixel the model gives a weed *probability*. We group pixels into 16x16 grid cells,
measure what fraction of each cell is weed, and bin into none/low/medium/high/severe.
This is the 'where is weed concentrated' output for herbicide targeting, not just yes/no.

In [ ]:
# reload best model
model = SemanticSegmentationTask.load_from_checkpoint(
    best_ckpt_path,
    model_factory=model.hparams.model_factory,
    model_args=model.hparams.model_args,
)
model.eval()

# one batch of predictions as PROBABILITIES (not hard 0/1)
test_loader = datamodule.test_dataloader()
with torch.no_grad():
    batch = next(iter(test_loader))
    images = batch["image"].to(model.device)
    masks = batch["mask"].numpy()
    outputs = model(images)
    probs = torch.softmax(outputs.output, dim=1)[:, 1, :, :].cpu().numpy()  # weed prob per pixel

# binning settings (tune BINS once you have real ground truth)
CELL = 16
THRESH = 0.5
BINS = [0.0, 0.05, 0.20, 0.40, 0.70, 1.01]
BIN_LABELS = ["none", "low", "medium", "high", "severe"]
BIN_COLORS = ["#f7f7f7", "#fee08b", "#fdae61", "#f46d43", "#a50026"]

cmap = ListedColormap(BIN_COLORS)
norm = BoundaryNorm(list(range(len(BIN_LABELS) + 1)), cmap.N)

def density_grid(prob_map):
    H, W = prob_map.shape
    weed = (prob_map >= THRESH).astype(float)
    gh, gw = H // CELL, W // CELL
    grid = np.zeros((gh, gw))
    for i in range(gh):
        for j in range(gw):
            block = weed[i*CELL:(i+1)*CELL, j*CELL:(j+1)*CELL]
            grid[i, j] = np.digitize(block.mean(), BINS) - 1
    return grid

N = min(5, images.shape[0])
fig, axes = plt.subplots(N, 3, figsize=(11, 3.4 * N))
if N == 1:
    axes = axes[None, :]

for k in range(N):
    img = images[k].cpu().numpy()
    rgb = np.stack([img[2], img[1], img[0]], axis=-1)  # R,G,B from bands 2,1,0
    rgb = (rgb - rgb.min()) / (rgb.max() - rgb.min() + 1e-8)

    axes[k, 0].imshow(rgb); axes[k, 0].set_title("HLS image"); axes[k, 0].axis("off")
    axes[k, 1].imshow(masks[k], cmap="Reds", vmin=0, vmax=1); axes[k, 1].set_title("True mask"); axes[k, 1].axis("off")
    grid = density_grid(probs[k])
    im = axes[k, 2].imshow(grid, cmap=cmap, norm=norm, interpolation="nearest")
    axes[k, 2].set_title("Weed intensity"); axes[k, 2].axis("off")

cbar = fig.colorbar(im, ax=axes, fraction=0.025, pad=0.02,
                    ticks=[x + 0.5 for x in range(len(BIN_LABELS))])
cbar.ax.set_yticklabels(BIN_LABELS)
plt.show()